# Thyroid Ultrasound: Binary Classification (Malignant vs Non-Malignant)

This notebook loads a Pascal VOC-style ultrasound dataset, extracts image crops from annotated bounding boxes, engineers features, and trains a Logistic Regression classifier.

**Dataset structure assumed:**
```
data/
  images/          ← .jpg ultrasound images
  annotations/     ← .xml Pascal VOC annotation files
  train.txt        ← image IDs for training
  val.txt          ← image IDs for validation
  test.txt         ← image IDs for test
```

The `<n>` field in each XML encodes the class label (e.g., `0` = benign, `1` = malignant).

## 1. Imports & Setup

In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import matplotlib.patches as patches
from pathlib import Path

from PIL import Image
from skimage.feature import graycomatrix, graycoprops
from skimage import img_as_ubyte
from skimage.color import rgb2gray

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

print("All imports successful.")

## 2. Configuration — Set Your Paths Here

In [ ]:

# Root of your dataset
# DATA_DIR = Path("add here")

# Subfolders
IMAGE_DIR      = DATA_DIR / "JPEGImages"
ANNOTATION_DIR = DATA_DIR / "Annotations"

# Splits
TRAIN_TXT = DATA_DIR / "ImageSets" / "Main" / "train.txt"
VAL_TXT   = DATA_DIR / "ImageSets" / "Main" / "val.txt"
TEST_TXT  = DATA_DIR / "ImageSets" / "Main" / "test.txt"
# Class mapping: adjust if your dataset uses different label integers
CLASS_MAP = {0: "benign", 1: "malignant"}

print(f"Images    : {IMAGE_DIR}")
print(f"Annotations: {ANNOTATION_DIR}")

print(IMAGE_DIR.exists())
print(ANNOTATION_DIR.exists())
print(TRAIN_TXT.exists())

## 3. Parse Annotations (Pascal VOC XML)

In [ ]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    filename = root.findtext("filename")
    width = int(root.findtext("size/width"))
    height = int(root.findtext("size/height"))

    objects = []
    for obj in root.findall("object"):
        label = int(obj.findtext("name"))   # <-- correct for your XML
        bb = obj.find("bndbox")

        objects.append({
            "label": label,
            "xmin": int(bb.findtext("xmin")),
            "ymin": int(bb.findtext("ymin")),
            "xmax": int(bb.findtext("xmax")),
            "ymax": int(bb.findtext("ymax")),
        })

    return {
        "filename": filename,
        "width": width,
        "height": height,
        "objects": objects
    }
    


def load_split(txt_file, image_dir, annotation_dir):
    ids = Path(txt_file).read_text().strip().splitlines()
    records = []

    for img_id in ids:
        img_id = img_id.strip()
        xml_path = annotation_dir / f"{img_id}.xml"
        img_path = image_dir / f"{img_id}.jpg"

        if not xml_path.exists() or not img_path.exists():
            #print(f"Missing file for {img_id}")
            continue

        try:
            ann = parse_xml(xml_path)
        except Exception as e:
            print(f"Error parsing {xml_path}: {e}")
            continue

        if len(ann["objects"]) == 0:
            print(f"No objects found in {xml_path}")

        for obj in ann["objects"]:
            records.append({
                "img_id": img_id,
                "img_path": str(img_path),
                "label": obj["label"],
                "xmin": obj["xmin"],
                "ymin": obj["ymin"],
                "xmax": obj["xmax"],
                "ymax": obj["ymax"],
            })

    return pd.DataFrame(records)


df_train = load_split(TRAIN_TXT, IMAGE_DIR, ANNOTATION_DIR)
df_val   = load_split(VAL_TXT,   IMAGE_DIR, ANNOTATION_DIR)
df_test  = load_split(TEST_TXT,  IMAGE_DIR, ANNOTATION_DIR)

print(f"Train samples : {len(df_train)}")
print(f"Val samples   : {len(df_val)}")
print(f"Test samples  : {len(df_test)}")
print(df_train.head())

ids = Path(TRAIN_TXT).read_text().strip().splitlines()

num_total = len(ids)
num_valid = 0

for img_id in ids:
    img_id = img_id.strip()
    if (IMAGE_DIR / f"{img_id}.jpg").exists() and (ANNOTATION_DIR / f"{img_id}.xml").exists():
        num_valid += 1

print("Train IDs listed:", num_total)
print("Train IDs available:", num_valid)
print("Missing:", num_total - num_valid)

def count_available(txt_file, image_dir, annotation_dir):
    ids = Path(txt_file).read_text().strip().splitlines()
    total = len(ids)
    available = 0

    for img_id in ids:
        img_id = img_id.strip()
        if (image_dir / f"{img_id}.jpg").exists() and (annotation_dir / f"{img_id}.xml").exists():
            available += 1

    print(f"{txt_file.name}: listed={total}, available={available}, missing={total-available}")

count_available(TRAIN_TXT, IMAGE_DIR, ANNOTATION_DIR)
count_available(VAL_TXT, IMAGE_DIR, ANNOTATION_DIR)
count_available(TEST_TXT, IMAGE_DIR, ANNOTATION_DIR)

## 4. Explore the Data

In [ ]:
# Class distribution
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (df, title) in zip(axes, [(df_train, "Train"), (df_val, "Val"), (df_test, "Test")]):
    counts = df["label"].map(CLASS_MAP).value_counts()
    counts.plot(kind="bar", ax=ax, color=["steelblue", "tomato"], edgecolor="k")
    ax.set_title(title)
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=0)
plt.suptitle("Class Distribution per Split", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualise a few samples with their bounding boxes
def show_samples(df, n=6, title="Samples"):
    sample = df.sample(min(n, len(df)), random_state=22)
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = Image.open(row["img_path"]).convert("RGB")
        ax.imshow(img, cmap="gray")
        rect = patches.Rectangle(
            (row["xmin"], row["ymin"]),
            row["xmax"] - row["xmin"],
            row["ymax"] - row["ymin"],
            linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.add_patch(rect)
        ax.set_title(CLASS_MAP.get(row["label"], row["label"]), fontsize=10)
        ax.axis("off")
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

show_samples(df_train, n=2, title="Training Samples with Bounding Boxes")

## 5. Feature Extraction from ROI Crops

We crop each annotated bounding box from its image and compute:
- **Intensity statistics**: mean, std, min, max, median (grayscale)
- **Texture (GLCM)**: contrast, dissimilarity, homogeneity, energy, correlation
- **Shape**: aspect ratio, area, relative area (fraction of image)

In [ ]:
def extract_features(row):
    """
    Crop the bounding box from the image and return a feature vector.
    """
    img = Image.open(row["img_path"]).convert("RGB")
    img_w, img_h = img.size

    # Crop the nodule ROI
    roi = img.crop((row["xmin"], row["ymin"], row["xmax"], row["ymax"]))
    roi = roi.resize((64, 64))                    # normalise size
    roi_gray = np.array(roi.convert("L"))         # uint8 grayscale

    # ── Intensity features ──────────────────────────────────────────────────
    feats = {
        "mean_intensity"   : roi_gray.mean(),
        "std_intensity"    : roi_gray.std(),
        "min_intensity"    : roi_gray.min().astype(float),
        "max_intensity"    : roi_gray.max().astype(float),
        "median_intensity" : np.median(roi_gray),
    }

    # ── GLCM texture features ────────────────────────────────────────────────
    glcm = graycomatrix(roi_gray, distances=[1], angles=[0], levels=256,
                        symmetric=True, normed=True)
    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]:
        feats[f"glcm_{prop}"] = graycoprops(glcm, prop)[0, 0]

    # ── Shape features ───────────────────────────────────────────────────────
    w = row["xmax"] - row["xmin"]
    h = row["ymax"] - row["ymin"]
    feats["aspect_ratio"]  = w / max(h, 1)
    feats["bbox_area"]     = w * h
    feats["relative_area"] = (w * h) / (img_w * img_h)

    return feats


def build_feature_matrix(df):
    rows = [extract_features(row) for _, row in df.iterrows()]
    X = pd.DataFrame(rows)
    y = df["label"].values
    return X, y


print("Extracting training features...")
X_train, y_train = build_feature_matrix(df_train)

print("Extracting validation features...")
X_val, y_val = build_feature_matrix(df_val)

print("Extracting test features...")
X_test, y_test = build_feature_matrix(df_test)

print(f"\nFeature matrix shape (train): {X_train.shape}")
X_train.describe()

## 6. Train Logistic Regression 



In [ ]:
# Pipeline: StandardScaler → LogisticRegression
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(
                   max_iter=1000,
                   class_weight="balanced",   # handles class imbalance
                   random_state=42
               ))
])

pipeline.fit(X_train, y_train)
print("Model trained.")

## 7. Evaluate on Validation Set

In [ ]:
y_val_pred  = pipeline.predict(X_val)
y_val_proba = pipeline.predict_proba(X_val)[:, 1]

print("── Validation Results ──")
print(f"Accuracy : {accuracy_score(y_val, y_val_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_val, y_val_proba):.3f}")
print()
print(classification_report(y_val, y_val_pred,
                             target_names=[CLASS_MAP[0], CLASS_MAP[1]]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_val, y_val_pred,
    display_labels=[CLASS_MAP[0], CLASS_MAP[1]],
    ax=axes[0], colorbar=False
)
axes[0].set_title("Confusion Matrix (Validation)")

# ROC curve
fpr, tpr, _ = roc_curve(y_val, y_val_proba)
auc = roc_auc_score(y_val, y_val_proba)
axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve (Validation)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
coef = pipeline.named_steps["clf"].coef_[0]
feat_names = X_train.columns.tolist()

importance_df = pd.DataFrame({
    "feature"    : feat_names,
    "coefficient": coef
}).sort_values("coefficient", key=abs, ascending=False)

plt.figure(figsize=(10, 5))
colors = ["tomato" if c > 0 else "steelblue" for c in importance_df["coefficient"]]
plt.barh(importance_df["feature"], importance_df["coefficient"], color=colors, edgecolor="k")
plt.axvline(0, color="k", linewidth=0.8)
plt.xlabel("Coefficient (scaled)")
plt.title("Logistic Regression Feature Importance")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

importance_df

## 9. Final Evaluation on Test Set

In [ ]:
y_test_pred  = pipeline.predict(X_test)
y_test_proba = pipeline.predict_proba(X_test)[:, 1]

print("── Test Results ──")
print(f"Accuracy : {accuracy_score(y_test, y_test_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_test_proba):.3f}")
print()
print(classification_report(y_test, y_test_pred,
                             target_names=[CLASS_MAP[0], CLASS_MAP[1]]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_test_pred,
    display_labels=[CLASS_MAP[0], CLASS_MAP[1]],
    ax=axes[0], colorbar=False
)
axes[0].set_title("Confusion Matrix (Test)")

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
auc = roc_auc_score(y_test, y_test_proba)
axes[1].plot(fpr, tpr, lw=2, color="steelblue", label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate (Recall)")
axes[1].set_title("ROC Curve (Test) — Logistic Regression")
axes[1].legend()

plt.tight_layout()
plt.show()